# 🐍 Master Guide: The GIL, Bytecode & Performance Optimization
> **Single-Source Revision Guide**: Covers bytecode disassembly (`dis`), opcode dispatch, Free-threaded Python (PEP 703 nogil), and performance tuning.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
Understanding bytecode interpretation and the Global Interpreter Lock (GIL) is crucial for performance tuning. Python 3.13+ introduces experimental free-threading (PEP 703) to disable the GIL.

---
## 🧠 2. CPython Architecture: Bytecode Disassembly & PEP 703

### Inspecting Bytecode with `dis.dis`
CPython compiles Python statements into stack-based bytecode instructions executed in the central dispatch loop in `Python/ceval.c`.

### PEP 703: Making the GIL Optional (Free-Threaded CPython)
To remove the GIL safely without crashing reference counters, PEP 703 uses:
1. **Biased Reference Counting**: Fast local counting for thread-owning objects.
2. **Mimalloc Memory Allocator**: Thread-safe memory allocation.
3. **Immortal Objects**: Flags on objects like `None`, `True`, `False` whose reference counts are never modified.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Disassembling Functions with `dis.dis`

In [ ]:
import dis

def calculate(a, b):
    return a * 2 + b

print("Bytecode disassembly for calculate():")
dis.dis(calculate)

---
## 🟡 4. Part 2: Intermediate Features

### Practical High-Performance Python Tips
1. **Local Variables (`LOAD_FAST`)**: Assigning global functions to local variables inside tight loops speeds up lookups from dict hashes to stack index arrays.
2. **Built-in `str.join()`**: $O(n)$ linear string concatenation vs $O(n^2)$ quadratic `+=`.
3. **List Comprehensions (`LIST_APPEND`)**: Inlined C bytecode vs method lookup `.append()`.

In [ ]:
import time

# Benchmark: Local variable cache in tight loop
import math

def slow_loop():
    return [math.sqrt(i) for i in range(100_000)]

def fast_loop():
    # Binding math.sqrt locally replaces LOAD_GLOBAL with LOAD_FAST:
    sqrt = math.sqrt
    return [sqrt(i) for i in range(100_000)]

t0 = time.perf_counter(); slow_loop(); t1 = time.perf_counter()
t2 = time.perf_counter(); fast_loop(); t3 = time.perf_counter()

print(f"slow_loop (global lookup): {(t1 - t0)*1000:.2f} ms")
print(f"fast_loop (local cache)  : {(t3 - t2)*1000:.2f} ms")

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| dis.dis(func)                      | Disassembles function into bytecode   |
| LOAD_FAST vs LOAD_GLOBAL           | Local vars are array-indexed (faster!)|
| PEP 703 (Python 3.13+)             | Optional free-threaded nogil build    |
| Immortal Objects                   | Unchanged ref counts (None, True, etc)|
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Memory Optimization: Standard `__dict__` vs `__slots__`
> **Core Principle**:
> - Standard class instances store attributes dynamically in a per-instance dictionary (`__dict__`), requiring ~150+ bytes of dictionary hash table overhead per instance.
> - Defining `__slots__` allocates fixed C-level struct pointers for attributes, saving ~60% memory and speeding up attribute lookups significantly.

In [ ]:
# Memory Footprint Benchmark: __dict__ vs __slots__
import sys

class StandardStudent:
    def __init__(self, sno, sname, marks):
        self.sno = sno
        self.sname = sname
        self.marks = marks

class SlottedStudent:
    __slots__ = ("sno", "sname", "marks")
    def __init__(self, sno, sname, marks):
        self.sno = sno
        self.sname = sname
        self.marks = marks

std_obj = StandardStudent(101, "Kavitha", 91.5)
slot_obj = SlottedStudent(101, "Kavitha", 91.5)

std_size = sys.getsizeof(std_obj) + sys.getsizeof(std_obj.__dict__)
slot_size = sys.getsizeof(slot_obj)

print(f"Standard Object + __dict__ Size: {std_size} bytes")
print(f"__slots__ Object Size           : {slot_size} bytes")
print(f"Memory Savings                  : {((std_size - slot_size) / std_size)*100:.1f}%")
print(f"Slotted object has no __dict__  : hasattr(slot_obj, '__dict__') = {hasattr(slot_obj, '__dict__')}")

### 2. Bytecode Analysis Drill (`dis.dis` Comparison)
Inspecting how the compiler optimizes list comprehensions vs manual `.append()` method calls.

In [ ]:
# Bytecode Comparison Drill
import dis

def comprehension_squares(n):
    return [x * x for x in range(n)]

def manual_append_squares(n):
    result = []
    for x in range(n):
        result.append(x * x)
    return result

print("=== Comprehension Bytecode (Uses specialized LIST_APPEND opcode) ===")
dis.dis(comprehension_squares)

print("\n=== Manual Loop Bytecode (Requires LOAD_METHOD & CALL_METHOD overhead) ===")
dis.dis(manual_append_squares)